# 03 - Feature Engineering

Compute phenological metrics from NDVI time-series, aggregate weather, create interaction terms, and PCA.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data_loader import load_merged_data
from src.features import (
    compute_phenological_metrics,
    aggregate_weather_features,
    create_interaction_terms,
    pca_on_spectral_bands,
    build_feature_matrix
)
from src.utils import get_data_dir

try:
    df = load_merged_data()
except FileNotFoundError:
    from src.synthetic_data import generate_synthetic_dataset
    df = generate_synthetic_dataset()
    get_data_dir().mkdir(parents=True, exist_ok=True)
    df.to_csv(get_data_dir() / 'merged_data.csv', index=False)

df['date'] = pd.to_datetime(df['date'])

## 1. Phenological Metrics

In [ ]:
pheno = compute_phenological_metrics(df)
print('Phenological metrics:')
print(pheno.describe())

## 2. Weather Aggregates

In [ ]:
weather = aggregate_weather_features(df)
print(weather.head())

## 3. Interaction Terms & PCA

In [ ]:
X_df, y = build_feature_matrix(df, include_pca=True, include_interactions=True)
feature_cols = [c for c in X_df.columns if c != 'field_id']
print(f'Features: {feature_cols}')
X_df.head()

In [ ]:
# Save features for modeling
out = X_df.copy()
out['yield'] = y.values
out.to_csv(get_data_dir() / 'features.csv', index=False)
print(f'Saved features to {get_data_dir() / "features.csv"}')